## GPT prompting: n20 examples, second filter

### requires python >= 3.10

### takes only the "no" answers previous result and runs through all three prompts and saves each result into individual file

In [1]:
# for auto-reloading extenrnal modules
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from tqdm import tqdm
import openai 
import os
from openai import AzureOpenAI
import configparser
import json
import csv
import sys

In [3]:
sys.path.append("../../../")
from common_code.gpt_utils import *
from common_code.gpt_reply_formats import *

In [63]:

from prompts.semantic_categories.v03.prompt import (
    ABSTRACT_SYSTEM_PROMPT, ABSTRACT_FEW_SHOTS_STR, ABSTRACT_FEW_SHOTS, 
)


from prompts.semantic_categories.v04.prompt import (
    ALIVE_SYSTEM_PROMPT, ALIVE_FEW_SHOTS_STR, ALIVE_FEW_SHOTS, 
    EVENT_SYSTEM_PROMPT, EVENT_FEW_SHOTS_STR, EVENT_FEW_SHOTS, 
    TIME_SYSTEM_PROMPT, TIME_FEW_SHOTS_STR, TIME_FEW_SHOTS,
    ORG_SYSTEM_PROMPT, ORG_FEW_SHOTS_STR, ORG_FEW_SHOTS
)

In [5]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

In [99]:
RESULTS_DIR = "../../results/"

EXAMPLE_FILE = RESULTS_DIR + "n20_examples_large_v01/gpt_v01/" + "gpt_b10_run01.csv"

GPT_ANSWER_FILE_ALIVE = RESULTS_DIR + "n20_examples_large_v01/gpt_v02/"+ "gpt_b10_run01_no_is_alive.csv"
GPT_ANSWER_FILE_EVENT = RESULTS_DIR + "n20_examples_large_v01/gpt_v02/"+ "gpt_b10_run01_no_is_event.csv"
GPT_ANSWER_FILE_TIME = RESULTS_DIR + "n20_examples_large_v01/gpt_v02/"+ "gpt_b10_run01_no_is_timex.csv"
GPT_ANSWER_FILE_ORG = RESULTS_DIR + "n20_examples_large_v01/gpt_v02/"+ "gpt_b10_run01_no_is_org.csv"
GPT_ANSWER_FILE_ABSTRACT = RESULTS_DIR + "n20_examples_large_v01/gpt_v02/" + "gpt_b10_run01_no_is_abstract.csv"

# fail ainult "no" vastustega (alive+event+time+org kokku panduna)
GPT_FILTERED_FILE = RESULTS_DIR + "n20_examples_large_v01/gpt_v02/"+ "gpt_b10_run01_no_filtered.csv"

# fail kõigi vastustega (kõik laused, tekitatud uus classification2 veerg)
GPT_ANSWER_FILE = RESULTS_DIR + "n20_examples_large_v01/gpt_v02/"+ "gpt_b10_run01.csv"

# väike sample fail 100 näitega
GPT_ANSWER_FILE_SAMP = RESULTS_DIR + "n20_examples_large_v01/gpt_v02/"+ "gpt_b10_run01_sample.csv"

CONF_FILE = "../../../../v04_verb-case_pattern/minu_code/azure.ini"


# OSA I : Andmed


## testimise põhjusel on kasutusel vana 10k v1 andmefail, et tulemusi saaks võrrelda

In [8]:
df1 = pd.read_csv(EXAMPLE_FILE, encoding="utf-8",  sep=",")

In [9]:
# võtta need, mille puhul gpt ütles "no"

spatial_obl_ex = df1[df1["classification"]=="no"]#[20:30]

In [10]:
spatial_obl_ex#[20:30]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
0,7108634,11432411,7,tasuma,NaN,ad,kord,korral,"“ Ei tea , kas esimesel korral tasub lõpuni minna , ” kahtlesin .",NaN,NaN,NaN,no,"The phrase 'korral' refers to 'occasion' and does not indicate a location, hence it is not adverbial of place."
1,2601827,4175718,12,pakkuma,NaN,ad,juhatus,juhatusel,"Eesti Filharmoonia Kammerkoor ja Tallinna Kammerorkester pakuvad täna õhtul Tõnu Kaljuste juhatusel Tallinna Metodisti kirikus toimuval kontserdil valiku teostest , mida koor ja orkester esitavad reedel algaval kolmenädalasel USA ja Kanada turneel .",NaN,NaN,NaN,no,"The phrase 'juhatusel' refers to 'under the direction' and is describing leadership or guidance, not location, so it is not adverbial of place."
2,1677278,2670999,3,seletama,NaN,all,patsient,patsiendile,"Kullamaa seletab patsiendile alati , et iga protees on organismile võõrkeha ja ümber selle tekitab organism poole aasta jooksul sidekoelise kapsli .",NaN,NaN,NaN,no,NaN
3,750912,1196732,14,virutama,NaN,all,reisija,reisijale,"Kaks aastat tagasi oli Eesti Päevalehe ajakirjanik tunnistajaks , kuidas kontrollija piletita sõitnud reisijale jalaga vastu tagumikku virutas ja vihaselt sõimles .",NaN,alive,NaN,no,NaN
4,4733683,7604499,14,lülitama,NaN,ad,poolaeg,poolajal,""" Seda ma paraku ei tea , sest lülitasin teleka sisse alles teisel poolajal . """,NaN,time,NaN,no,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9990,12758554,20409273,5,meenuma,NaN,all,Andrew,Andrew'le,Uue plaadi tulekuga meenub Andrew'le veel üks probleem .,NaN,NaN,PER,no,"The phrase 'Andrew'le' refers to a person receiving something, but it does not describe a location, so it is not adverbial of place."
9994,1715798,2731307,14,lõpetama,NaN,el,isa,isast,"Ma otsutasin , et kui lööb ette aasta 2000 , lõpetan ma oma isast rääkimise .",NaN,alive,NaN,no,NaN
9995,9799023,15722082,8,käsitlema,NaN,ad,päev,päevil,Film käsitleb sündmusi 1943. aastal Varssavis kannatusnädala päevil ja põhineb Jerzy Andrzejewski romaanil .,NaN,NaN,NaN,no,NaN
9997,12893712,20625029,4,saama,sisse,ad,esitamine,esitamisel,Riigikogulased saavad töötõendi esitamisel tasuta sisse .,NaN,NaN,NaN,no,NaN


# OSA II : GPT

## GPT jaoks vajalik

In [7]:
config = configparser.ConfigParser()

status = config.read(CONF_FILE) 
assert status == [CONF_FILE]

API_VERSION = config['azure-configuration']['api_version']
AZURE_ENDPOINT = config['azure-configuration']['api_base']
SUBSCRIPTION_KEY = config['azure-configuration']['api_key']
model_name = "gpt-4o" #"GPT-4o-2024-1120 Global"
DEPLOYMENT = config['azure-configuration']['deployment_id']

In [8]:
client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=SUBSCRIPTION_KEY,
)

## Andmete söötmine

In [9]:

def classify_batch(my_batch, few_shots, system_prompt, client, deployment):
    """Gets a yes/no answer for a batch of sentences and phrases. 
    """
    #print("classify", len(my_batch))
    max_att = 1
    attempt = 0
    while attempt < max_att:
        attempt += 1
        user_payload = {
            "few_shots": few_shots,
            "batch": my_batch
        }
    
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":  json.dumps(user_payload, ensure_ascii=False)}
        ]

        #return None, None
        response = client.chat.completions.create(
            model=deployment,
            messages=messages,
            temperature=0, # absoluutselt min väljund 
        )

        raw_output = response.choices[0].message.content.strip()

        try:
            data = json.loads(raw_output)

            if len(data) != len(my_batch):
                raise ValueError(f"Väljundis ei ole õige arv vastuseid. Peaks olema {len(batch)} aga on {len(data)}.")
                
            elif len(data) == len(my_batch):
                for item in data:
                    ClassificationDict(**item)

            return response, raw_output

        except (ValidationError, json.JSONDecodeError, ValueError) as e:
            #print(f"Attempt {attempt} failed. Retrying batch...")
            print(f"Error: {e}")
            #print(f"Raw output: {raw_output[:500]}...")  # preview first 500 chars
            time.sleep(1)  # small delay before retry

    print(f"Batch failed after {max_att} attempts.")
    # isegi kui ei saanud kõike kätte siis saab pärast äkki käsitsi midagi juurde panna
    return response, raw_output


# ABSTRACT LOCATION

In [24]:
df_0 = spatial_obl_ex.copy()

## NB! muuda max_allowed_tok kui vaja

In [26]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0
# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 1500000

rows = df_0.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append(json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, ABSTRACT_FEW_SHOTS_STR, ABSTRACT_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

1it [00:00,  1.63it/s]


In [ ]:
used_tokens

In [1]:
len(results)

In [28]:
if len(results) == len(df_0):
    df_0["is_abstract"] = [r["a"] for r in results]

In [29]:
df_0.to_csv(GPT_ANSWER_FILE_ABSTRACT, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# ALIVE

## NB! muuda max_allowed_tok kui vaja

In [14]:
#df = spatial_obl_ex.sample(frac=1)#.reset_index(drop=True)
df_1 = df_0[df_0["is_abstract"]=="no"].copy()

In [16]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0

# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 1500000

rows = df_1.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append( json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, ALIVE_FEW_SHOTS_STR, ALIVE_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

565it [06:39,  1.41it/s]


In [17]:
used_tokens # 10 lauset batch 10-> 1200 tokenit, 565 batchi x 10 lauset -> 627,414 tokenit

627414

In [18]:
len(results)

5647

## andmed tabelisse 


In [19]:
if len(results) == len(df_1):
    df_1["is_alive"] = [r["a"] for r in results]

/tmp/ipykernel_10151/3079534175.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_1["is_alive"] = [r["a"] for r in results]


In [20]:
df_1.to_csv(GPT_ANSWER_FILE_ALIVE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# TIMEX

In [21]:
df_3 = df_1[df_1["is_alive"]=="no"].copy()

## NB! muuda max_allowed_tok kui vaja

In [22]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0
# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 1500000

rows = df_3.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append(json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, TIME_FEW_SHOTS_STR, TIME_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

565it [06:21,  1.48it/s]


In [23]:
used_tokens # 10 lauset, batch 10 -> 1200 tokenit,  565 batchi x 10 lauset -> 603,100 tokenit

603100

In [24]:
len(results)

5647

In [25]:
if len(results) == len(df_3):
    df_3["is_time"] = [r["a"] for r in results]

/tmp/ipykernel_10151/2180909303.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_3["is_time"] = [r["a"] for r in results]


In [26]:
df_3.to_csv(GPT_ANSWER_FILE_TIME, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# EVENT

In [28]:
df_2 = df_3[df_3["is_time"]=="no"].copy()

## NB! muuda max_allowed_tok kui vaja

In [29]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0
# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 1500000

rows = df_2.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append(json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, EVENT_FEW_SHOTS_STR, EVENT_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

565it [06:24,  1.47it/s]


In [30]:
used_tokens # 10 lauset batch 10-> 1600 tokenit, 565 batchi x 10 lauset -> 837,983 tokenit

837983

In [31]:
len(results)

5647

In [32]:
if len(results) == len(df_2):
    df_2["is_event"] = [r["a"] for r in results]

/tmp/ipykernel_10151/3742971824.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2["is_event"] = [r["a"] for r in results]


In [33]:
df_2.to_csv(GPT_ANSWER_FILE_EVENT, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# ORG

In [ ]:
df_4 = df_2[df_2["is_event"]=="no"].copy()

In [66]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0
# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 900000

rows = df_4.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append(json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, ORG_FEW_SHOTS_STR, ORG_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

255it [02:44,  1.55it/s]


In [51]:
used_tokens

293072

In [52]:
len(results)

2543

In [67]:
if len(results) == len(df_4):
    df_4["is_org"] = [r["a"] for r in results]

/tmp/ipykernel_30041/3092172851.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_4["is_org"] = [r["a"] for r in results]


In [74]:
df_4.to_csv(GPT_ANSWER_FILE_ORG, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

### Kokku alive+event+time filtreeritud fail

In [80]:
fname1 = GPT_ANSWER_FILE_ABSTRACT 
fname2 = GPT_ANSWER_FILE_EVENT
fname3 = GPT_ANSWER_FILE_TIME
fname4 = GPT_ANSWER_FILE_ORG
fname5 = GPT_ANSWER_FILE_ALIVE

df1 = pd.read_csv(fname1, encoding="utf-8",  sep=",")
df2 = pd.read_csv(fname2, encoding="utf-8",  sep=",")
df3 = pd.read_csv(fname3, encoding="utf-8",  sep=",")
df4 = pd.read_csv(fname4, encoding="utf-8",  sep=",")
df5 = pd.read_csv(fname5, encoding="utf-8",  sep=",")

key_cols = ['sentence_id','head_id', "verb", "verb_compound", "morph_case", "form"]

df2_selected = df2[key_cols + ["is_event"]].copy()
df3_selected = df3[key_cols + ["is_time"]].copy()
df4_selected = df4[key_cols + ["is_org"]].copy()
df5_selected = df5[key_cols + ["is_alive"]].copy()

df1['verb_compound'] = df1['verb_compound'].astype('string').str.strip()
df2_selected['verb_compound'] = df2_selected['verb_compound'].astype('string').str.strip()
df3_selected['verb_compound'] = df3_selected['verb_compound'].astype('string').str.strip()
df4_selected['verb_compound'] = df4_selected['verb_compound'].astype('string').str.strip()
df5_selected['verb_compound'] = df5_selected['verb_compound'].astype('string').str.strip()

# Merge df1 with df2_selected etc
merged_df = df1.merge(df2_selected, on=key_cols, how='left')

merged_df2 = merged_df.merge(df3_selected, on=key_cols, how='left')

merged_df3 = merged_df2.merge(df4_selected, on=key_cols, how='left')

final_df = merged_df3.merge(df5_selected, on=key_cols, how='left')

cols = ['is_time', 'is_event', 'is_alive', 'is_org', 'is_abstract']

final_df[cols] = final_df[cols].fillna('no')
final_df['verb_compound'] = final_df['verb_compound'].fillna('')

In [82]:
final_df.to_csv(GPT_FILTERED_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

## Kokku kõikide varasemate klassifikatsioonidega + classification3 loomine

### seda osa saab korrata ilma gpt osa uuesti tegemata

In [83]:
df1 = pd.read_csv(EXAMPLE_FILE, encoding="utf-8", sep=",")
df2 = pd.read_csv(GPT_FILTERED_FILE, encoding="utf-8", sep=",")

In [86]:
cols = ["head_id", "form", "verb", "verb_compound", "morph_case","sentence_id", "is_time", "is_alive", "is_event", "is_org", "is_abstract"]
df3 = df2[cols]

In [87]:
filter3 = pd.merge(df1, df3, on=["head_id", "form", "verb", "verb_compound", "morph_case", "sentence_id"], how='left')
filter3

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org
0,7108634,11432411,7,tasuma,NaN,ad,kord,korral,"“ Ei tea , kas esimesel korral tasub lõpuni minna , ” kahtlesin .",NaN,NaN,NaN,no,"The phrase 'korral' refers to 'occasion' and does not indicate a location, hence it is not adverbial of place.",no,no,no,no
1,2601827,4175718,12,pakkuma,NaN,ad,juhatus,juhatusel,"Eesti Filharmoonia Kammerkoor ja Tallinna Kammerorkester pakuvad täna õhtul Tõnu Kaljuste juhatusel Tallinna Metodisti kirikus toimuval kontserdil valiku teostest , mida koor ja orkester esitavad reedel algaval kolmenädalasel USA ja Kanada turneel .",NaN,NaN,NaN,no,"The phrase 'juhatusel' refers to 'under the direction' and is describing leadership or guidance, not location, so it is not adverbial of place.",no,no,no,no
2,1677278,2670999,3,seletama,NaN,all,patsient,patsiendile,"Kullamaa seletab patsiendile alati , et iga protees on organismile võõrkeha ja ümber selle tekitab organism poole aasta jooksul sidekoelise kapsli .",NaN,NaN,NaN,no,NaN,no,yes,no,no
3,750912,1196732,14,virutama,NaN,all,reisija,reisijale,"Kaks aastat tagasi oli Eesti Päevalehe ajakirjanik tunnistajaks , kuidas kontrollija piletita sõitnud reisijale jalaga vastu tagumikku virutas ja vihaselt sõimles .",NaN,alive,NaN,no,NaN,no,yes,no,no
4,4733683,7604499,14,lülitama,NaN,ad,poolaeg,poolajal,""" Seda ma paraku ei tea , sest lülitasin teleka sisse alles teisel poolajal . """,NaN,time,NaN,no,NaN,yes,no,yes,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9799023,15722082,8,käsitlema,NaN,ad,päev,päevil,Film käsitleb sündmusi 1943. aastal Varssavis kannatusnädala päevil ja põhineb Jerzy Andrzejewski romaanil .,NaN,NaN,NaN,no,NaN,yes,no,yes,no
9996,4898982,7865678,13,jääma,kõrvale,el,seanss,seanssidest,"Kui aga tegemist on abikaasade omavaheliste probleemidega , jäävad lapsed reeglina neist seanssidest kõrvale .",NaN,NaN,NaN,yes,"The phrase 'seanssidest' describes a location or context from which someone is excluded, thus classified as adverbial of place.",NaN,NaN,NaN,NaN
9997,12893712,20625029,4,saama,sisse,ad,esitamine,esitamisel,Riigikogulased saavad töötõendi esitamisel tasuta sisse .,NaN,NaN,NaN,no,NaN,no,no,yes,no
9998,523151,824406,6,võitma,NaN,ad,järv,Järvel,"Üksikpiltidest võitis Hermes Sarapuu "" Järvel "" .",NaN,NaN,LOC,yes,"The phrase 'Järvel' (on the lake) specifies a location, making it adverbial of place.",NaN,NaN,NaN,NaN


In [89]:
# kas saab klassifitseerimisel korraks kõrvale jätta
# see on yes/no selleks, et lihtsamalt näha, kas peale eeldefineeritud klasside saab veel midagi välja võtta

def exclude(row):
    
    # kui eelmine tulemus = "yes" -> jääb "yes"  -> saame välja visata
    if row["classification"] == "yes":
        return "yes"
    
    # kui on aeg -> "yes" -> saame välja visata
    if row["is_time"] == "yes":
        return "yes"
    
    # event -> "yes" -> saame välja visata
    if row["is_event"] == "yes":
        return "yes"
    
    # elus -> "yes" -> saame välja visata
    if row["is_alive"] == "yes":
        return "yes"
    
    # org -> "yes" -> saame välja visata
    if row["is_org"] == "yes":
        return "yes"

    
    # kui oli "no", NaN ja/või alive/time/abstract kõik olid "no"
    else:
        return "no"

In [90]:
filter3["exclude"] = filter3.apply(exclude, axis=1)

In [95]:
# uus classification2
# muuta vastavalt vajadusele

def new_class(row):
    
    # kui eelmine tulemus = "yes" -> jääb "yes"  -> saame välja visata
    if row["classification"] == "yes":
        return "loc"
    
    if row["is_abstract"] == "yes":
        return "loc"
    
    # kui on aeg -> "yes" -> saame välja visata
    if row["is_time"] == "yes":
        return "time"
    
    # event -> "yes" -> saame välja visata
    if row["is_event"] == "yes":
        return "event"
    
    # elus -> "yes" -> saame välja visata
    if row["is_alive"] == "yes":
        return "actor"
    
    # org -> "yes" -> saame välja visata
    if row["is_org"] == "yes":
        return "actor"

    
    # kui oli "no", NaN ja/või alive/time/abstract kõik olid "no"
    else:
        return "UNK"

In [96]:
filter3["classification2"] = filter3.apply(new_class, axis=1)

In [100]:
filter3.to_csv(GPT_ANSWER_FILE, encoding="utf-8", index=False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [101]:
filter3_2 = filter3.iloc[:100]

In [102]:
filter3_2.to_csv(GPT_ANSWER_FILE_SAMP, encoding="utf-8", index=False, sep=",", quoting=csv.QUOTE_MINIMAL)